# 04 — Тюнинг, PCA, калибровка, порог, интерпретируемость

Этот ноутбук собирает все эксперименты CP2:
1. CV-сравнение моделей (mean ± std).
2. RandomizedSearchCV для 4 кандидатов.
3. PCA для линейной модели.
4. Калибровка вероятностей.
5. Подбор порога под cost-функционал.
6. Интерпретируемость финальной модели.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.config import settings
from src.cv import cv_results_to_frame, cv_score_models
from src.dim_reduction import explained_variance_curve, pca_results_to_frame, run_pca_experiment
from src.interpret import permutation_feature_importance, tree_feature_importance
from src.modeling import _build_models, make_train_test
from src.threshold import (
    best_threshold_by_cost,
    best_threshold_by_f1,
    evaluate_calibration,
    threshold_grid_to_frame,
)
from src.tuning import run_tuning, tuning_results_to_frame

settings.images_dir.mkdir(parents=True, exist_ok=True)
X_train, X_test, y_train, y_test = make_train_test(use_fe=True)
len(X_train), len(X_test)

## 1. CV-сравнение моделей

Гипотеза: ансамбли деревьев устойчивее линейных моделей по ROC-AUC.
Метод: 5-fold StratifiedKFold на train, скор — ROC-AUC.

In [ ]:
models = _build_models(X_train)
cv_results = cv_score_models(models, X_train, y_train)
cv_df = cv_results_to_frame(cv_results)
cv_df

## 2. Гиперпараметрический поиск

Гипотеза: дефолтные XGB/LGBM не оптимальны — RandomizedSearchCV (n_iter=15, 5-fold) подберёт лучше.
Метод: для LogReg+FE / RF / XGBoost / LightGBM.

In [ ]:
tuning = run_tuning(n_iter=15, save=True)
tuning_df = tuning_results_to_frame(tuning).sort_values('roc_auc', ascending=False)
tuning_df

In [ ]:
print(json.dumps({k: v.best_params for k, v in tuning.items()}, indent=2, ensure_ascii=False))

## 3. PCA для линейной модели

Гипотеза: после OHE+FE признаков много (~70+), декорреляция через PCA может помочь LogReg.
Метод: explained variance curve + LogReg(PCA=10/20/30/0.95) на test.

In [ ]:
ev = explained_variance_curve(X_train, max_components=60)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ev['k'], ev['cumulative_explained_variance'], marker='o', markersize=3)
ax.axhline(0.95, ls='--', color='red', label='95%')
ax.set_xlabel('k компонент'); ax.set_ylabel('Кумулятивная объяснённая дисперсия')
ax.set_title('PCA explained variance')
ax.legend(); fig.tight_layout()
fig.savefig(settings.images_dir / 'pca_explained_variance.png', dpi=120)
plt.show()

In [ ]:
pca_results = run_pca_experiment(X_train, y_train, X_test, y_test, components=(10, 20, 30, 0.95))
pca_df = pca_results_to_frame(pca_results)
pca_df

## 4. Калибровка вероятностей

Гипотеза: XGBoost даёт хороший ranking, но вероятности шумные → калибровка снизит Brier.
Метод: CalibratedClassifierCV (isotonic) на CV-фолдах train.

In [ ]:
import joblib
best_pipe = joblib.load(settings.models_dir / 'tuned_xgboost.joblib')
calib = evaluate_calibration(best_pipe, X_train, y_train, X_test, y_test, method='isotonic')
print(calib)

## 5. Подбор порога

Гипотеза: порог 0.5 не оптимален при cost FN=5, FP=1 — пропущенный дефолт дороже.
Метод: сетка 0.05..0.95, выбираем минимум cost; альтернатива — максимум F1.

In [ ]:
best_pipe.fit(X_train, y_train)
proba = best_pipe.predict_proba(X_test)[:, 1]
grid = threshold_grid_to_frame(y_test.values, proba)
best_cost = best_threshold_by_cost(y_test.values, proba)
best_f1 = best_threshold_by_f1(y_test.values, proba)
print('best by cost:', best_cost)
print('best by f1  :', best_f1)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(grid['threshold'], grid['precision'], label='precision')
ax.plot(grid['threshold'], grid['recall'], label='recall')
ax.plot(grid['threshold'], grid['f1'], label='f1')
ax.axvline(best_cost.threshold, color='red', ls='--', label=f'best cost @ {best_cost.threshold:.2f}')
ax.set_xlabel('threshold'); ax.set_ylabel('value'); ax.legend(); ax.set_title('Threshold scan')
fig.tight_layout(); fig.savefig(settings.images_dir / 'threshold_scan.png', dpi=120); plt.show()

## 6. Интерпретируемость

Tree feature importance — что внутри XGBoost; permutation importance — что реально двигает скор на test.

In [ ]:
tree_imp = tree_feature_importance(best_pipe, top_k=20)
tree_imp

In [ ]:
perm_imp = permutation_feature_importance(best_pipe, X_test, y_test, n_repeats=5, top_k=15)
fig, ax = plt.subplots(figsize=(6, 4.5))
perm_imp.iloc[::-1].plot.barh(x='feature', y='importance', xerr='std', ax=ax, color='#4a7ab5', legend=False)
ax.set_xlabel('Δ ROC-AUC при перемешивании')
ax.set_title('Permutation importance (test)')
fig.tight_layout(); fig.savefig(settings.images_dir / 'permutation_importance.png', dpi=120)
plt.show()